# Experiment 7: Implementation of Decision Tree on Real-World Dataset




### Step 1: Import Libraries & Load Wine Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

### Step 2: Explore Dataset & Split

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

print("Dataset Shape:", X.shape)
print("Target Classes:", list(wine.target_names))
display(X.head())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

### Step 3: Analyze Pre-Pruning (`max_depth`) & Compare Gini vs. Entropy

In [ ]:
depths = range(1, 11)
train_acc_gini, test_acc_gini = [], []
train_acc_ent, test_acc_ent = [], []

for d in depths:
    dt_g = DecisionTreeClassifier(criterion='gini', max_depth=d, random_state=42).fit(X_train, y_train)
    dt_e = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc_gini.append(accuracy_score(y_train, dt_g.predict(X_train)))
    test_acc_gini.append(accuracy_score(y_test, dt_g.predict(X_test)))
    train_acc_ent.append(accuracy_score(y_train, dt_e.predict(X_train)))
    test_acc_ent.append(accuracy_score(y_test, dt_e.predict(X_test)))

plt.figure(figsize=(8.5, 4.8))
plt.plot(depths, train_acc_gini, 'o--', color='#4F46E5', label='Train Acc (Gini)')
plt.plot(depths, test_acc_gini, 'o-', color='#0D9488', lw=2.2, label='Test Acc (Gini)')
plt.plot(depths, test_acc_ent, 's-', color='#D97706', lw=2.2, label='Test Acc (Entropy)')
plt.title('Effect of max_depth Pruning on Overfitting', fontweight='bold')
plt.xlabel('Tree Max Depth'); plt.ylabel('Accuracy'); plt.xticks(depths)
plt.legend(); plt.grid(True, alpha=0.4); plt.show()

### Step 4: Train Final Pruned Decision Tree & Visualize Tree + Feature Importances

In [ ]:
best_tree = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
best_tree.fit(X_train, y_train)
y_pred = best_tree.predict(X_test)

print(f"Pruned Decision Tree Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=wine.target_names))

# Visualize the Decision Tree
plt.figure(figsize=(15, 8))
plot_tree(best_tree, feature_names=wine.feature_names, class_names=wine.target_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Trained Decision Tree (Entropy, max_depth=3)', fontsize=14, fontweight='bold')
plt.show()

# Feature Importances
imp = pd.Series(best_tree.feature_importances_, index=wine.feature_names).sort_values()
plt.figure(figsize=(8, 5))
imp[imp > 0].plot(kind='barh', color='#4F46E5', edgecolor='#1E1B4B')
plt.title('Decision Tree Feature Importances', fontweight='bold')
plt.xlabel('Importance Score')
plt.grid(True, axis='x', alpha=0.4)
plt.show()

### Conclusion
- Decision Trees naturally handle non-linear rules and require zero feature scaling.
- Pre-pruning (`max_depth=3`) eliminates overfitting and yields an interpretable flowchart identifying key chemical features like `flavanoids` and `proline`.